# Build a flood-extent layer from a 3Di depth raster

This notebook turns one timestep of a 3Di depth raster into polygons, one feature per depth band. Each step is one cell, so you can run it a cell at a time and inspect the result in QGIS after any step.

The notebook first reclassifies the depth in meters into a raster that holds 0 for depths under 5 cm, a band value (10 to 60, and 600) for wet cells, and nodata for dry cells. It needs GDAL 3.3 or later and GRASS GIS 8 on the command line.

## Set up

Set the 3Di depth raster and the band to use, the working folder, and the settings that control the result. Each band of a 3Di depth raster is one timestep, and `TIMESTEP_BAND` counts from 1.

- `RADIUS` is the closing radius in cells, which are meters at this raster's 1 m resolution.
- `MIN_AREA` is the smallest hole or polygon to keep, in square meters.

`WORK` holds every intermediate file and the GRASS project, so you can open them in QGIS after any step. Rerunning this cell deletes the GRASS project and the intermediate layers, and leaves other files in `WORK` alone. The helpers run GRASS modules inside that project and print the time each step takes.

In [ ]:
import subprocess
import shutil
import time
from contextlib import contextmanager
from pathlib import Path

DEPTH_TIF = Path(
    "/Users/maianhtran/Documents/3Di/Case-study-sim-center/test-code/Test event/"
    "recreate event 20251007/depth-analysis/depth_event20251007_Ver03_rain-csv/"
    "depth_event20251007_Ver03.tif"
)
TIMESTEP_BAND = 1
DEPTH_BANDS = Path("../../artifacts/docs/experiment_depth-raster-geojson-polygon/re-classify_depth.tif").resolve()
OUT_DIR = Path("../../artifacts/docs/experiment_depth-raster-geojson-polygon").resolve()
RADIUS = 8
MIN_AREA = 20

WORK = OUT_DIR
WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK / "grass_project"
shutil.rmtree(PROJECT, ignore_errors=True)
print("Working folder:", WORK)


def fresh(path):
    path.unlink(missing_ok=True)
    return path


@contextmanager
def timed(label):
    start = time.time()
    yield
    print(f"{label}: {time.time() - start:.1f} s")


def run(*args):
    subprocess.run([str(a) for a in args], check=True)


def grass(module, *args):
    run("grass", PROJECT / "PERMANENT", "--exec", module, *args, "--overwrite", "--quiet")


def band_counts(raster):
    out = subprocess.run(
        ["grass", str(PROJECT / "PERMANENT"), "--exec", "r.stats", "-c", "-n", f"input={raster}"],
        check=True, capture_output=True, text=True,
    ).stdout
    print(out)

Working folder: /Users/maianhtran/Documents/3Di/Case-study-sim-center/airflow_clone/t3-digital-twin/artifacts/docs/experiment_depth-raster-geojson-polygon


## Reclassify the depth raster

`reclassify_depth` turns the depth in meters into integer depth codes. It reads band `TIMESTEP_BAND` of `DEPTH_TIF` and writes `DEPTH_BANDS`.

`BAND_TABLE` lists the upper depth limit of each band and the code that band gets. A depth up to 0.05 m gets 0. The next bands end at 0.1, 0.2, 0.3, 0.4, 0.5, and 0.6 m and get the codes 10, 20, 30, 40, 50, and 60, which are the upper limits in centimeters. A depth above 0.6 m gets `TOP_CODE`. A depth exactly on a limit belongs to the lower band.

Nodata cells stay nodata. The output is `Int16` with nodata -9999 and DEFLATE compression.

In [13]:
BAND_TABLE = [(0.05, 0), (0.1, 10), (0.2, 20), (0.3, 30), (0.4, 40), (0.5, 50), (0.6, 60)]
TOP_CODE = 600
DEPTH_NODATA = -9999


def reclassify_depth(src, dst, band=1, table=BAND_TABLE, top_code=TOP_CODE):
    expression = str(top_code)
    for limit, code in reversed(table):
        expression = f"numpy.where(A<={limit},{code},{expression})"
    run("gdal_calc.py", "-A", src, f"--A_band={band}", f"--outfile={dst}",
        f"--calc={expression}", f"--NoDataValue={DEPTH_NODATA}", "--type=Int16",
        "--co=COMPRESS=DEFLATE", "--overwrite", "--quiet")


with timed("reclassify depth"):
    reclassify_depth(DEPTH_TIF, DEPTH_BANDS, TIMESTEP_BAND)

reclassify depth: 3.6 s


## Close the bands and export the raster

This step imports the raster into GRASS, closes the small gaps between wet cells, and exports the result as `final.tif`.

1. `grass -c` creates a GRASS project in `WORK` that copies the raster's CRS (VN-2000 / UTM zone 48N). The import keeps the raster's own grid, and `g.region` sets the working region to it.
2. Keep only the wet cells. `r.grow.distance` measures the distance to the nearest cell that isn't null. The 0 cells are valid but not wet, so they become null here. Otherwise every cell counts as a source and the grow step changes nothing.
3. Grow the bands by `RADIUS` cells. Every empty cell within `RADIUS` cells of a wet cell takes the band of that nearest wet cell. Isolated wet cells touch and merge into blobs. Raise `RADIUS` if isolated cells still don't merge.
4. Shrink the bands back. Remove every grown cell within `RADIUS` cells of a cell that is still empty. This undoes the growth at the outer edge and keeps the merged interior, which is a morphological closing.
5. Put back the 0 cells that the closing didn't cover, so the output keeps all 8 values of the input. Delete the `final` expression if you don't want 0 in the polygons. You then get 7 values.
6. Print the band counts before and after. The values match, and cells move between bands only where closing filled a gap or merged a blob.
7. Export as `Int16` with nodata -9999. A `Byte` export would force 0 to be nodata and lose the 0 band. Open `final.tif` in QGIS now if you want to check the closed mask before polygonizing.

In [14]:
FINAL_TIF = WORK / "final.tif"

with timed("import"):
    run("grass", "-c", DEPTH_BANDS, PROJECT, "--exec", "r.in.gdal",
        f"input={DEPTH_BANDS}", "output=depth_bands", "-o", "--quiet")
    grass("g.region", "raster=depth_bands")
band_counts("depth_bands")

with timed("close bands"):
    grass("r.mapcalc", "expression=wet_bands = if(depth_bands > 0, int(depth_bands), null())")
    grass("r.grow.distance", "input=wet_bands", "distance=dist1", "value=val1")
    grass("r.mapcalc",
          f"expression=dilated = if(!isnull(wet_bands), wet_bands, if(dist1 <= {RADIUS}, int(val1), null()))")
    grass("r.mapcalc", "expression=inv = if(isnull(dilated), 1, null())")
    grass("r.grow.distance", "input=inv", "distance=dist2")
    grass("r.mapcalc", f"expression=closed = if(dist2 <= {RADIUS}, null(), dilated)")
    grass("r.mapcalc",
          "expression=final = if(!isnull(closed), closed, if(depth_bands == 0, 0, null()))")
band_counts("final")

with timed("export raster"):
    run("grass", PROJECT / "PERMANENT", "--exec", "r.out.gdal", "input=final",
        f"output={FINAL_TIF}", "format=GTiff", "type=Int16", "nodata=-9999",
        "createopt=COMPRESS=LZW", "--overwrite", "--quiet")

Creating new GRASS GIS project <grass_project>...


import: 1.0 s
0 1109188
10 857640
20 1483993
30 1240873
40 1037377
50 759335
60 645873
600 3364048

close bands: 16.0 s
0 686947
10 1722478
20 1698598
30 1288633
40 1055661
50 769039
60 651068
600 3375767



ERROR 6: final.tif, band 1: SetColorTable() only supported for Byte or UInt16 bands in TIFF format.


export raster: 1.1 s


## Polygonize

`-8` merges diagonally touching cells of the same band into one polygon. Expect several hundred thousand features at this point. Most are small patches.

In [15]:
POLY = fresh(WORK / "poly.gpkg")
with timed("polygonize"):
    run("gdal_polygonize.py", "-q", "-8", FINAL_TIF, "-f", "GPKG", POLY, "poly", "DN")

polygonize: 2.0 s


## Keep the wet polygons and dissolve by band

Keep only `DN` above 0, then merge all polygons of the same band into one multipolygon. This takes about a minute. It cuts the layer from several hundred thousand features to 7, one per band, which makes every later step cheaper.

In [16]:
DISSOLVED = fresh(WORK / "dissolved.gpkg")
with timed("dissolve"):
    run("ogr2ogr", "-q", "-f", "GPKG", DISSOLVED, POLY, "-dialect", "SQLite",
        "-sql", "SELECT DN, ST_Union(geom) AS geom FROM poly WHERE DN > 0 GROUP BY DN",
        "-nln", "dissolved")

dissolve: 63.2 s


## Delete small holes and specks

Closing and dissolve leave small holes inside a band and small polygons outside it. `clean_multipolygon` removes every hole and every polygon with an area below `MIN_AREA`, and `clean_layer` applies it to each band. The layer is for display, we will fix the min area = 20 m².

In [17]:
from osgeo import ogr

ogr.UseExceptions()


def ring_area(ring):
    polygon = ogr.Geometry(ogr.wkbPolygon)
    polygon.AddGeometry(ring)
    return polygon.GetArea()


def clean_multipolygon(geom, min_area):
    cleaned = ogr.Geometry(ogr.wkbMultiPolygon)
    for part in geom:
        kept = ogr.Geometry(ogr.wkbPolygon)
        kept.AddGeometry(part.GetGeometryRef(0))
        for i in range(1, part.GetGeometryCount()):
            hole = part.GetGeometryRef(i)
            if ring_area(hole) >= min_area:
                kept.AddGeometry(hole)
        if kept.GetArea() >= min_area:
            cleaned.AddGeometry(kept)
    return cleaned


def clean_layer(src, dst, min_area):
    src_ds = ogr.Open(str(src))
    layer = src_ds.GetLayer()
    definition = layer.GetLayerDefn()
    Path(dst).unlink(missing_ok=True)
    dst_ds = ogr.GetDriverByName("GPKG").CreateDataSource(str(dst))
    out = dst_ds.CreateLayer("cleaned", srs=layer.GetSpatialRef(), geom_type=ogr.wkbMultiPolygon)
    for i in range(definition.GetFieldCount()):
        out.CreateField(definition.GetFieldDefn(i))
    for feature in layer:
        geom = clean_multipolygon(ogr.ForceToMultiPolygon(feature.GetGeometryRef().Clone()), min_area)
        if geom.IsEmpty():
            continue
        cleaned = ogr.Feature(out.GetLayerDefn())
        cleaned.SetFrom(feature)
        cleaned.SetGeometry(geom)
        out.CreateFeature(cleaned)


CLEANED = WORK / "cleaned.gpkg"
with timed("clean"):
    clean_layer(DISSOLVED, CLEANED, MIN_AREA)

clean: 1.4 s


## Cut vector tiles

The app draws the extent from vector tiles, because the cleaned layer is too large to load into the browser as one GeoJSON file. `ogr2ogr -f MVT` reprojects to Web Mercator and writes one tile per XYZ address into `TILES_DIR`. Tiles beyond `MAX_ZOOM` are stretched by the map, so the tiles stay sharp at closer views. `MAX_SIZE` and `MAX_FEATURES` are raised because the default 500 kB limit makes GDAL coarsen the densest tiles.


In [ ]:
MIN_ZOOM = 10
MAX_ZOOM = 16
TILES_DIR = WORK / "mvt"

shutil.rmtree(TILES_DIR, ignore_errors=True)
with timed("cut tiles"):
    run("ogr2ogr", "-q", "-f", "MVT", TILES_DIR, CLEANED,
        "-dsco", "NAME=flood_extent", "-dsco", f"MINZOOM={MIN_ZOOM}", "-dsco", f"MAXZOOM={MAX_ZOOM}",
        "-dsco", "COMPRESS=NO", "-dsco", "SIMPLIFICATION=0",
        "-dsco", "MAX_SIZE=5000000", "-dsco", "MAX_FEATURES=1000000",
        "-dsco", f"TEMPORARY_DB={WORK / 'mvt_tmp.sqlite'}")
print(TILES_DIR)
